# LeetCode 339 — Nested List Weight Sum bằng Python

**Link đề:** https://leetcode.com/problems/nested-list-weight-sum/
**Độ khó:** Easy (theo danh mục của repo) · **Chủ đề:** Depth-First Search, Breadth-First Search, Recursion · **Bài Premium**

> Ghi chú: LeetCode hiện ghi bài này là **Medium**; notebook lưu ở `DSA/Easy` theo danh mục của repo.

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
from typing import List

# """
# This is the interface that allows for creating nested lists.
# You should not implement it, or speculate about its implementation
# """
# class NestedInteger:
#     def isInteger(self) -> bool: ...
#     def getInteger(self) -> int: ...
#     def getList(self) -> List[NestedInteger]: ...

class Solution:
    def depthSum(self, nestedList: List[NestedInteger]) -> int:
        ...
```

## Mục tiêu học
1. Hiểu cấu trúc **danh sách lồng nhau** (nested list) và khái niệm **độ sâu** (depth).
2. Tự dựng class `NestedInteger` giống LeetCode để chạy notebook độc lập.
3. Nắm **đệ quy** (recursion): trường hợp cơ sở, lời gọi con, **call stack** — và viết **DFS** truyền tham số `depth`.
4. Viết thêm **BFS theo tầng** (dùng `collections.deque`) và **DFS bằng stack tường minh**.
5. Trace từng bước, chứng minh, phân tích Time/Space, và giải biến thể **Nested List Weight Sum II (364)**.

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Cho một danh sách lồng nhau các số nguyên `nestedList`. Mỗi phần tử **hoặc là một số nguyên**, **hoặc là một danh sách** mà phần tử của nó lại có thể là số nguyên hoặc danh sách khác.

**Độ sâu** của một số nguyên là số danh sách bao quanh nó. Ví dụ trong `[1, [2, 2], [[3], 2], 1]`, mỗi số có giá trị bằng đúng độ sâu của nó.

Trả về **tổng của mọi số nguyên nhân với độ sâu của nó**.

| Input | Output | Giải thích |
|---|---:|---|
| `[[1,1],2,[1,1]]` | `10` | bốn số `1` ở độ sâu 2, số `2` ở độ sâu 1: `4·1·2 + 2·1 = 10` |
| `[1,[4,[6]]]` | `27` | `1·1 + 4·2 + 6·3 = 27` |
| `[0]` | `0` | `0·1 = 0` |

**Constraints (theo hiểu biết về đề):** `1 <= nestedList.length <= 50`; giá trị các số nguyên trong `[-100, 100]`; độ sâu tối đa của số nguyên `<= 50`.

In [1]:
# Dùng list Python thường để nhìn cấu trúc trước khi dựng NestedInteger
example = [1, [4, [6]]]
print(example[0])          # 1       -> độ sâu 1
print(example[1])          # [4, [6]]
print(example[1][0])       # 4       -> độ sâu 2
print(example[1][1][0])    # 6       -> độ sâu 3

1
[4, [6]]
4
6


## 2. Nền tảng Python cho bài này

### 2.1. `isinstance(x, int)` / `isinstance(x, list)` — kiểm tra kiểu

Khi phần tử có thể là "số hoặc danh sách", ta cần hỏi "nó là gì?" trước khi xử lý.

In [2]:
for item in [1, [2, 3], [], -5]:
    print(repr(item), "| là int?", isinstance(item, int), "| là list?", isinstance(item, list))

1 | là int? True | là list? False
[2, 3] | là int? False | là list? True
[] | là int? False | là list? True
-5 | là int? True | là list? False


### 2.2. Dựng class `NestedInteger` giống giao diện LeetCode

LeetCode không đưa list Python thường mà đưa **list các đối tượng `NestedInteger`**. Mỗi đối tượng có:
- `isInteger()` → `True` nếu đang giữ một số.
- `getInteger()` → số đó (hoặc `None` nếu là list).
- `getList()` → list con các `NestedInteger` (hoặc `None` nếu là số).

Ta tự cài đặt class này + hàm `build(...)` chuyển list Python thường sang `list[NestedInteger]`, và hàm `show(...)` để in lại.

In [3]:
from typing import List, Optional

class NestedInteger:
    def __init__(self, value: Optional[int] = None):
        # value = None -> danh sách rỗng; ngược lại -> một số nguyên
        self._integer = value
        self._list = [] if value is None else None

    def isInteger(self) -> bool:
        return self._integer is not None

    def getInteger(self) -> Optional[int]:
        return self._integer

    def setInteger(self, value: int) -> None:
        self._integer, self._list = value, None

    def add(self, elem: "NestedInteger") -> None:
        if self._list is None:          # đang là số -> chuyển thành list
            self._integer, self._list = None, []
        self._list.append(elem)

    def getList(self) -> Optional[List["NestedInteger"]]:
        return self._list


def build_one(x) -> NestedInteger:
    if isinstance(x, int):
        return NestedInteger(x)
    node = NestedInteger()
    for child in x:
        node.add(build_one(child))
    return node

def build(py_list) -> List[NestedInteger]:
    # LeetCode truyền vào list các NestedInteger (tầng ngoài cùng)
    return [build_one(x) for x in py_list]

def show(nested: List[NestedInteger]) -> str:
    def one(ni):
        return str(ni.getInteger()) if ni.isInteger() else "[" + ",".join(one(c) for c in ni.getList()) + "]"
    return "[" + ",".join(one(ni) for ni in nested) + "]"

nl = build([[1, 1], 2, [1, 1]])
print(show(nl))
print(nl[1].isInteger(), nl[1].getInteger())
print(nl[0].isInteger(), len(nl[0].getList()))

[[1,1],2,[1,1]]
True 2
False 2


### 2.3. Đệ quy (recursion) và call stack

**Đệ quy** là hàm tự gọi lại chính nó với một bài toán **nhỏ hơn**. Cần:
1. **Trường hợp cơ sở** (base case): dừng, không gọi tiếp.
2. **Bước đệ quy**: chia nhỏ và gọi lại.

Mỗi lời gọi được đặt lên **call stack**; lời gọi con trả về thì lời gọi cha tiếp tục. Ví dụ đếm tổng một list lồng nhau (chưa nhân độ sâu):

In [4]:
def plain_sum(x, indent=0):
    pad = "  " * indent
    if isinstance(x, int):                       # base case
        print(f"{pad}số {x}")
        return x
    print(f"{pad}vào list {x}")
    total = sum(plain_sum(child, indent + 1) for child in x)   # bước đệ quy
    print(f"{pad}ra list {x} -> {total}")
    return total

print("Tổng:", plain_sum([1, [4, [6]]]))

vào list [1, [4, [6]]]
  số 1
  vào list [4, [6]]
    số 4
    vào list [6]
      số 6
    ra list [6] -> 6
  ra list [4, [6]] -> 10
ra list [1, [4, [6]]] -> 11
Tổng: 11


### 2.4. `collections.deque` — hàng đợi cho BFS

`deque` cho phép thêm vào cuối (`append`) và lấy ra ở đầu (`popleft`) trong `O(1)`. (`list.pop(0)` tốn `O(n)`.)

In [5]:
from collections import deque
q = deque([1, 2])
q.append(3)
print(q.popleft(), q.popleft(), list(q))   # 1 2 [3]

1 2 [3]


## 3. Cách trực giác — "làm phẳng" kèm độ sâu rồi cộng

Ý tưởng dễ nghĩ: đi qua cấu trúc, thu mọi cặp `(số, độ sâu)` vào một list phẳng, rồi tính `sum(số * độ sâu)`. Đây là cách tách bài toán thành 2 bước dễ kiểm tra.

In [6]:
def flatten_with_depth(nested: List[NestedInteger], depth: int = 1, out=None):
    if out is None:
        out = []
    for ni in nested:
        if ni.isInteger():
            out.append((ni.getInteger(), depth))
        else:
            flatten_with_depth(ni.getList(), depth + 1, out)
    return out

def depth_sum_flatten(nested: List[NestedInteger]) -> int:
    pairs = flatten_with_depth(nested)
    return sum(value * depth for value, depth in pairs)

nl = build([1, [4, [6]]])
print(flatten_with_depth(nl))
print(depth_sum_flatten(nl))

[(1, 1), (4, 2), (6, 3)]
27


## 4. Nút thắt

Cách làm phẳng **đúng**, nhưng:
- Tốn thêm bộ nhớ `O(N)` cho list `pairs` (N = số số nguyên) — không cần thiết vì ta chỉ cần **tổng**.
- Có thể cộng **ngay khi gặp** số nguyên, không cần lưu lại.

Câu hỏi then chốt: **làm sao biết độ sâu của một số khi gặp nó?** → Truyền `depth` như một **tham số** khi đi xuống: vào một list con thì `depth + 1`.

## 5. Insight — DFS mang theo độ sâu

```text
dfs(list, depth) = tổng over phần tử x trong list:
    nếu x là số   -> x * depth
    nếu x là list -> dfs(x.getList(), depth + 1)
đáp án = dfs(nestedList, 1)
```

- Tầng ngoài cùng có độ sâu **1** (không phải 0).
- Mỗi lần "chui" vào một list con, độ sâu tăng 1; khi lời gọi con trả về, ta tự động "quay về" độ sâu cũ (vì `depth` là biến cục bộ của mỗi lời gọi).

In [7]:
# Kiểm tra nhanh ý "depth là biến cục bộ của mỗi lời gọi"
def show_depths(x, depth=1):
    for item in x:
        if isinstance(item, int):
            print(f"số {item} ở depth {depth}")
        else:
            show_depths(item, depth + 1)
    # sau khi vòng for kết thúc, quay về lời gọi cha với depth của cha

show_depths([[1, 1], 2, [1, 1]])

số 1 ở depth 2
số 1 ở depth 2
số 2 ở depth 1
số 1 ở depth 2
số 1 ở depth 2


## 6. Lời giải chính — giảng từng đoạn rồi ghép

**Đoạn 1 — hàm con `dfs(items, depth)`:**
```python
def dfs(items, depth):
    total = 0
```

**Đoạn 2 — duyệt từng phần tử, hỏi "số hay list?":**
```python
    for ni in items:
        if ni.isInteger():
            total += ni.getInteger() * depth
        else:
            total += dfs(ni.getList(), depth + 1)
    return total
```

**Đoạn 3 — gọi từ tầng ngoài cùng với depth = 1:** `return dfs(nestedList, 1)`.

Base case ở đâu? Khi `items` không còn list con (hoặc rỗng), vòng `for` chỉ cộng số hoặc không làm gì → **không gọi đệ quy nữa**.

In [8]:
def depth_sum_dfs(nested_list: List[NestedInteger]) -> int:
    def dfs(items: List[NestedInteger], depth: int) -> int:
        total = 0
        for ni in items:
            if ni.isInteger():
                total += ni.getInteger() * depth
            else:
                total += dfs(ni.getList(), depth + 1)
        return total
    return dfs(nested_list, 1)

for data in [[[1, 1], 2, [1, 1]], [1, [4, [6]]], [0]]:
    print(data, "->", depth_sum_dfs(build(data)))

[[1, 1], 2, [1, 1]] -> 10
[1, [4, [6]]] -> 27
[0] -> 0


## 7. TRACE từng bước

Bảng tay cho `[1, [4, [6]]]`:

| Lời gọi | depth | Phần tử | Loại | Đóng góp | total của lời gọi |
|---|---:|---|---|---:|---:|
| dfs(ngoài) | 1 | `1` | số | 1·1 = 1 | 1 |
| dfs(ngoài) | 1 | `[4,[6]]` | list → gọi con | (chờ) | |
| dfs(`[4,[6]]`) | 2 | `4` | số | 4·2 = 8 | 8 |
| dfs(`[4,[6]]`) | 2 | `[6]` | list → gọi con | (chờ) | |
| dfs(`[6]`) | 3 | `6` | số | 6·3 = 18 | 18 → trả 18 |
| dfs(`[4,[6]]`) | 2 | | nhận 18 | | 8 + 18 = 26 → trả 26 |
| dfs(ngoài) | 1 | | nhận 26 | | 1 + 26 = **27** |

In [9]:
def trace_depth_sum(nested_list: List[NestedInteger]) -> int:
    def dfs(items, depth):
        pad = "    " * (depth - 1)
        print(f"{pad}-> vào dfs(depth={depth}) với {show(items)}")
        total = 0
        for ni in items:
            if ni.isInteger():
                total += ni.getInteger() * depth
                print(f"{pad}   số {ni.getInteger()} × {depth} = {ni.getInteger() * depth:>3} | total = {total}")
            else:
                sub = dfs(ni.getList(), depth + 1)
                total += sub
                print(f"{pad}   nhận {sub} từ list con | total = {total}")
        print(f"{pad}<- trả {total} (depth={depth})")
        return total
    return dfs(nested_list, 1)

print("Kết quả:", trace_depth_sum(build([1, [4, [6]]])))

-> vào dfs(depth=1) với [1,[4,[6]]]
   số 1 × 1 =   1 | total = 1
    -> vào dfs(depth=2) với [4,[6]]
       số 4 × 2 =   8 | total = 8
        -> vào dfs(depth=3) với [6]
           số 6 × 3 =  18 | total = 18
        <- trả 18 (depth=3)
       nhận 18 từ list con | total = 26
    <- trả 26 (depth=2)
   nhận 26 từ list con | total = 27
<- trả 27 (depth=1)
Kết quả: 27


In [10]:
print("Kết quả:", trace_depth_sum(build([[1, 1], 2, [1, 1]])))

-> vào dfs(depth=1) với [[1,1],2,[1,1]]
    -> vào dfs(depth=2) với [1,1]
       số 1 × 2 =   2 | total = 2
       số 1 × 2 =   2 | total = 4
    <- trả 4 (depth=2)
   nhận 4 từ list con | total = 4
   số 2 × 1 =   2 | total = 6
    -> vào dfs(depth=2) với [1,1]
       số 1 × 2 =   2 | total = 2
       số 1 × 2 =   2 | total = 4
    <- trả 4 (depth=2)
   nhận 4 từ list con | total = 10
<- trả 10 (depth=1)
Kết quả: 10


## 8. Các lời giải khác

### Cách 3 — BFS theo tầng

Xử lý **từng tầng** một: hàng đợi chứa các phần tử cùng độ sâu. Số nguyên thì cộng `giá trị × depth`; list thì đẩy các con vào hàng đợi của tầng sau.

In [11]:
def depth_sum_bfs(nested_list: List[NestedInteger]) -> int:
    queue = deque(nested_list)
    depth = 1
    total = 0
    while queue:
        for _ in range(len(queue)):        # chỉ xử lý đúng các phần tử của tầng hiện tại
            ni = queue.popleft()
            if ni.isInteger():
                total += ni.getInteger() * depth
            else:
                queue.extend(ni.getList())
        depth += 1
    return total

print(depth_sum_bfs(build([[1, 1], 2, [1, 1]])), depth_sum_bfs(build([1, [4, [6]]])))

10 27


In [12]:
def trace_bfs(nested_list: List[NestedInteger]) -> int:
    queue = deque(nested_list)
    depth, total = 1, 0
    while queue:
        level = list(queue)
        level_ints = [ni.getInteger() for ni in level if ni.isInteger()]
        for _ in range(len(queue)):
            ni = queue.popleft()
            if ni.isInteger():
                total += ni.getInteger() * depth
            else:
                queue.extend(ni.getList())
        print(f"tầng {depth}: {[show([x])[1:-1] for x in level]} | số nguyên {level_ints} | total = {total}")
        depth += 1
    return total

trace_bfs(build([1, [4, [6]]]))

tầng 1: ['1', '[4,[6]]'] | số nguyên [1] | total = 1
tầng 2: ['4', '[6]'] | số nguyên [4] | total = 9
tầng 3: ['6'] | số nguyên [6] | total = 27


27

### Cách 4 — DFS bằng stack tường minh (không đệ quy)

Đẩy cặp `(phần_tử, depth)` vào stack. Hữu ích khi độ sâu rất lớn (tránh giới hạn đệ quy ~1000 của Python).

In [13]:
def depth_sum_stack(nested_list: List[NestedInteger]) -> int:
    stack = [(ni, 1) for ni in nested_list]
    total = 0
    while stack:
        ni, depth = stack.pop()
        if ni.isInteger():
            total += ni.getInteger() * depth
        else:
            for child in ni.getList():
                stack.append((child, depth + 1))
    return total

print(depth_sum_stack(build([[1, 1], 2, [1, 1]])), depth_sum_stack(build([1, [4, [6]]])))

10 27


## 9. Code nộp LeetCode

Trên LeetCode **không** định nghĩa lại class `NestedInteger` (đã có sẵn). Ta chỉ nộp `class Solution`.

### Cách chính — DFS đệ quy

In [14]:
class Solution:
    def depthSum(self, nestedList: List[NestedInteger]) -> int:
        def dfs(items: List[NestedInteger], depth: int) -> int:
            total = 0
            for ni in items:
                if ni.isInteger():
                    total += ni.getInteger() * depth
                else:
                    total += dfs(ni.getList(), depth + 1)
            return total
        return dfs(nestedList, 1)

print(Solution().depthSum(build([[1, 1], 2, [1, 1]])))

10


### Cách nộp thay thế — BFS theo tầng

In [15]:
class SolutionBFS:
    def depthSum(self, nestedList: List[NestedInteger]) -> int:
        queue = deque(nestedList)
        depth, total = 1, 0
        while queue:
            for _ in range(len(queue)):
                ni = queue.popleft()
                if ni.isInteger():
                    total += ni.getInteger() * depth
                else:
                    queue.extend(ni.getList())
            depth += 1
        return total

print(SolutionBFS().depthSum(build([1, [4, [6]]])))

27


## 10. Vì sao đúng?

**Mệnh đề (quy nạp theo độ sâu lồng nhau):** `dfs(items, d)` trả về `Σ (giá trị × độ sâu thật)` của mọi số nằm trong `items`, **với điều kiện** các phần tử trực tiếp của `items` có độ sâu thật là `d`.

- Phần tử là **số**: độ sâu thật là `d`, đóng góp `x · d` ✔.
- Phần tử là **list** `L`: các phần tử trực tiếp của `L` nằm trong thêm một lớp list → độ sâu `d + 1`. Theo giả thiết quy nạp (list con nông hơn), `dfs(L, d + 1)` trả đúng phần đóng góp của `L` ✔.
- Lời gọi đầu `dfs(nestedList, 1)`: các phần tử tầng ngoài nằm trong đúng 1 list → độ sâu 1 ✔.

**BFS:** invariant "đầu mỗi vòng `while`, hàng đợi chứa đúng các phần tử có độ sâu `depth`" — vì ta chỉ pop đúng `len(queue)` phần tử của tầng hiện tại, còn con của chúng (độ sâu `depth + 1`) được thêm vào sau.

## 11. Time / Space Complexity và so sánh

Gọi `N` = tổng số phần tử (số nguyên + list con) trong cấu trúc, `D` = độ sâu tối đa.

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| Làm phẳng + cộng | `O(N)` | `O(N)` list cặp + `O(D)` stack | dễ kiểm tra từng bước |
| **DFS đệ quy** | **`O(N)`** | **`O(D)`** call stack | gọn, tự nhiên nhất |
| BFS theo tầng | `O(N)` | `O(W)` với W = số phần tử nhiều nhất trên một tầng | duyệt theo tầng |
| DFS stack tường minh | `O(N)` | `O(N)` worst | tránh giới hạn đệ quy |

Mỗi phần tử được thăm **đúng một lần** → `O(N)`.

## 12. Bộ test — mẫu đề, biên, đối chiếu

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.depthSum(build([[1, 1], 2, [1, 1]])), 10)
check("đề 2", sol.depthSum(build([1, [4, [6]]])), 27)
check("đề 3", sol.depthSum(build([0])), 0)
check("số âm", sol.depthSum(build([-1, [-2]])), -1 - 4)
check("list rỗng lồng nhau", sol.depthSum(build([[], [[]], 5])), 5)
check("chỉ có list rỗng", sol.depthSum(build([[]])), 0)
check("mỗi số = độ sâu", sol.depthSum(build([1, [2, 2], [[3], 2], 1])), 1 + 4 + 4 + 9 + 4 + 1)

PASS đề 1: 10
PASS đề 2: 27
PASS đề 3: 0
PASS số âm: -5
PASS list rỗng lồng nhau: 5
PASS chỉ có list rỗng: 0
PASS mỗi số = độ sâu: 23


Test tiếp theo kiểm tra **độ sâu tối đa theo constraint (50)**: hàm `make_depth(5, 50)` bọc số 5 trong 49 lớp list rồi đặt vào list ngoài cùng → 5 nằm trong đúng 50 list → đóng góp `5 × 50 = 250`. Ví dụ nhỏ để tự kiểm: `make_depth(5, 3)` = `[[[5]]]`.

In [17]:
import random

def make_depth(value: int, depth: int):
    x = value
    for _ in range(depth - 1):     # depth - 1 lớp bên trong + 1 lớp ngoài cùng = depth
        x = [x]
    return [x]

check("đúng độ sâu 50", sol.depthSum(build(make_depth(5, 50))), 250)

def ref_depth_sum(py_list, depth=1):
    return sum(x * depth if isinstance(x, int) else ref_depth_sum(x, depth + 1) for x in py_list)

def random_nested(max_depth: int):
    items = []
    for _ in range(random.randint(0, 4)):
        if max_depth > 1 and random.random() < 0.4:
            items.append(random_nested(max_depth - 1))
        else:
            items.append(random.randint(-100, 100))
    return items

random.seed(339)
methods = [depth_sum_flatten, depth_sum_dfs, depth_sum_bfs, depth_sum_stack,
           Solution().depthSum, SolutionBFS().depthSum]
for _ in range(500):
    data = random_nested(6) or [0]
    expected = ref_depth_sum(data)
    nl = build(data)
    for f in methods:
        assert f(nl) == expected, (f, data)
print("500 cấu trúc ngẫu nhiên: 6 cách khớp với tham chiếu trên list Python thường")

PASS đúng độ sâu 50: 250
500 cấu trúc ngẫu nhiên: 6 cách khớp với tham chiếu trên list Python thường


## 13. Lỗi người mới hay gặp

### Lỗi 1: Bắt đầu với `depth = 0`
Tầng ngoài cùng có độ sâu 1. Bắt đầu từ 0 làm mọi số ở tầng ngoài bị nhân 0.

In [18]:
def depth_sum_start_zero(nested_list):
    def dfs(items, depth):
        return sum(ni.getInteger() * depth if ni.isInteger() else dfs(ni.getList(), depth + 1) for ni in items)
    return dfs(nested_list, 0)            # SAI

nl = build([[1, 1], 2, [1, 1]])
print("sai :", depth_sum_start_zero(nl), "| đúng:", depth_sum_dfs(nl))

sai : 4 | đúng: 10


### Lỗi 2: Tăng `depth` "toàn cục" mà không giảm lại
Dùng một biến chung `self.depth += 1` khi vào list con nhưng quên `-= 1` khi ra → các phần tử anh em phía sau bị tính sâu hơn thực tế.

In [19]:
class BuggyGlobalDepth:
    def depthSum(self, nestedList):
        self.depth = 1
        return self.dfs(nestedList)

    def dfs(self, items):
        total = 0
        for ni in items:
            if ni.isInteger():
                total += ni.getInteger() * self.depth
            else:
                self.depth += 1
                total += self.dfs(ni.getList())
                # QUÊN: self.depth -= 1
        return total

nl = build([[1, 1], 2, [1, 1]])
print("sai :", BuggyGlobalDepth().depthSum(nl), "| đúng:", depth_sum_dfs(nl))

sai : 14 | đúng: 10


### Lỗi 3: Gọi `getInteger()` trên phần tử là list
Trả về `None` → `None * depth` gây `TypeError`. Luôn hỏi `isInteger()` trước.

### Lỗi 4: BFS không giới hạn theo tầng
Viết `while queue: ni = queue.popleft(); ...; depth += 1` → `depth` tăng sau **mỗi phần tử** chứ không phải mỗi tầng.

In [20]:
try:
    ni_list = build([[1]])[0]
    print(ni_list.getInteger() * 2)
except TypeError as e:
    print("TypeError:", e)

def bfs_no_level(nested_list):
    queue = deque(nested_list)
    depth, total = 1, 0
    while queue:
        ni = queue.popleft()
        if ni.isInteger():
            total += ni.getInteger() * depth
        else:
            queue.extend(ni.getList())
        depth += 1                          # SAI: tăng theo phần tử
    return total

nl = build([[1, 1], 2, [1, 1]])
print("sai :", bfs_no_level(nl), "| đúng:", depth_sum_bfs(nl))

TypeError: unsupported operand type(s) for *: 'NoneType' and 'int'
sai : 26 | đúng: 10


## 14. Mẹo nhận dạng pattern

- **Cấu trúc tự lồng nhau** (list trong list, cây, thư mục trong thư mục) → **đệ quy / DFS**.
- Cần thông tin "đang ở tầng nào" → **truyền `depth` làm tham số** (đi xuống +1, tự động quay về khi return).
- Cần xử lý "theo từng tầng" → **BFS với `for _ in range(len(queue))`**.
- Độ sâu có thể rất lớn (> 1000) → chuyển sang **stack tường minh**.
- Đây là "cây tổng quát": mỗi list là một nút, các phần tử là nút con. Kỹ thuật giống hệt Maximum Depth of N-ary Tree (559).

## 15. Biến thể

### Biến thể 1 — Nested List Weight Sum II (LeetCode 364): trọng số đảo ngược
Trọng số = `maxDepth - depth + 1` (số nằm sâu nhất có trọng số 1). Mẹo BFS: cộng dồn **tổng chưa nhân** của các tầng (`running`) vào kết quả sau mỗi tầng — số ở tầng nông được cộng nhiều lần hơn. Cẩn thận: `maxDepth` tính theo **số nguyên** sâu nhất, nên các tầng cuối chỉ chứa list rỗng phải bỏ qua.

### Biến thể 2 — độ sâu tối đa của nested list

In [21]:
def depth_sum_inverse(nested_list: List[NestedInteger]) -> int:
    queue = deque(nested_list)
    level_sums = []          # tổng (chưa nhân) các số nguyên của từng tầng
    level_has_int = []       # tầng đó có số nguyên nào không
    while queue:
        s, has = 0, False
        for _ in range(len(queue)):
            ni = queue.popleft()
            if ni.isInteger():
                s += ni.getInteger()
                has = True
            else:
                queue.extend(ni.getList())
        level_sums.append(s)
        level_has_int.append(has)
    # maxDepth chỉ tính theo tầng sâu nhất CÓ số nguyên (bỏ các tầng chỉ chứa list rỗng)
    last = max((i for i, h in enumerate(level_has_int) if h), default=-1)
    running = 0              # tổng các số từ tầng 1 tới tầng hiện tại
    result = 0
    for i in range(last + 1):
        running += level_sums[i]
        result += running    # số ở tầng nông được cộng lại nhiều lần hơn = trọng số lớn hơn
    return result

def max_depth(nested_list: List[NestedInteger]) -> int:
    best = 0
    for ni in nested_list:
        best = max(best, 1 if ni.isInteger() else 1 + max_depth(ni.getList()))
    return best if nested_list else 1

print(depth_sum_inverse(build([[1, 1], 2, [1, 1]])))   # 8: bốn số 1 trọng số 1, số 2 trọng số 2
print(depth_sum_inverse(build([1, [4, [6]]])))         # 17: 1*3 + 4*2 + 6*1
print(max_depth(build([1, [4, [6]]])), max_depth(build([[1, 1], 2])))

8
17
3 2


## 16. Bài tập tự luyện (TODO — không chạy sẵn)

### Bài 1 — Tính tay
Tính `depthSum([2, [3, [4, 5]], [[1]]])` bằng bảng (giá trị, độ sâu, tích).

### Bài 2 — Dự đoán output
`show(build([[], 1]))` in gì? `len(build([[1, 2], 3]))` bằng bao nhiêu?

### Bài 3 — Điền chỗ trống

In [22]:
def exercise_depth_sum(nested_list: List[NestedInteger]) -> int:
    def dfs(items, depth):
        total = 0
        for ni in items:
            # TODO: nếu là số -> cộng giá trị * depth; nếu là list -> đệ quy với depth + 1
            pass
        return total
    return dfs(nested_list, 1)

# assert exercise_depth_sum(build([[1, 1], 2, [1, 1]])) == 10
# assert exercise_depth_sum(build([1, [4, [6]]])) == 27

### Bài 4 — Tự viết: đếm số lượng số nguyên ở mỗi độ sâu
Trả về dict `{depth: số lượng số nguyên}`.

In [23]:
def exercise_count_by_depth(nested_list: List[NestedInteger]) -> dict:
    # TODO
    pass

# assert exercise_count_by_depth(build([[1, 1], 2, [1, 1]])) == {1: 1, 2: 4}
# assert exercise_count_by_depth(build([1, [4, [6]]])) == {1: 1, 2: 1, 3: 1}

### Bài 5 — Biến thể: làm phẳng (Flatten) theo thứ tự xuất hiện
Trả về list các số nguyên theo thứ tự từ trái sang phải (ý tưởng của LeetCode 341).

In [24]:
def exercise_flatten(nested_list: List[NestedInteger]) -> list[int]:
    # TODO
    pass

# assert exercise_flatten(build([[1, 1], 2, [1, 1]])) == [1, 1, 2, 1, 1]
# assert exercise_flatten(build([1, [4, [6]]])) == [1, 4, 6]

### Bài 6 — Mini challenge: weight sum II bằng DFS hai lượt
Lượt 1 tìm `maxDepth` (chỉ tính các tầng **có số nguyên** — nếu cần). Lượt 2 cộng `giá trị × (maxDepth - depth + 1)`. So sánh với `depth_sum_inverse`.

In [25]:
def exercise_depth_sum_inverse_dfs(nested_list: List[NestedInteger]) -> int:
    # TODO 1: hàm get_max_depth(items, depth)
    # TODO 2: hàm dfs(items, depth, max_d)
    pass

# assert exercise_depth_sum_inverse_dfs(build([[1, 1], 2, [1, 1]])) == 8
# assert exercise_depth_sum_inverse_dfs(build([1, [4, [6]]])) == 17

## 17. Gợi ý 3 mức

**Bài 1** — (1) 2 ở độ sâu 1. (2) 3 ở độ sâu 2; 4, 5 ở độ sâu 3; 1 ở độ sâu 3. (3) Cộng các tích.

**Bài 2** — (1) `show` in lại cấu trúc. (2) List rỗng in là `[]`. (3) Tầng ngoài có 2 phần tử.

**Bài 3** — (1) Hỏi `isInteger()` trước. (2) `total += ni.getInteger() * depth`. (3) `total += dfs(ni.getList(), depth + 1)`.

**Bài 4** — (1) Truyền `depth` như DFS chính. (2) `counts[depth] = counts.get(depth, 0) + 1`. (3) Dùng dict chung bên ngoài hàm con.

**Bài 5** — (1) Gặp số thì `append`. (2) Gặp list thì đệ quy. (3) Thứ tự duyệt `for` giữ nguyên thứ tự trái → phải.

**Bài 6** — (1) `maxDepth` của `[1,[4,[6]]]` là 3. (2) Trọng số `max_d - depth + 1`. (3) Hai lượt đều `O(N)`.

Lưu ý bài 6: với input chứa list rỗng ở tầng sâu nhất (như `[1, [[]]]`), định nghĩa "maxDepth" có thể khác nhau; hãy chỉ tính độ sâu của **số nguyên** để khớp với BFS ở mục 15.

## 18. Đáp án tham khảo

**Bài 1:** `2·1 + 3·2 + 4·3 + 5·3 + 1·3 = 2 + 6 + 12 + 15 + 3 = 38`.

**Bài 2:** `[[],1]` và `2`.

In [26]:
def answer_depth_sum(nested_list):
    def dfs(items, depth):
        total = 0
        for ni in items:
            if ni.isInteger():
                total += ni.getInteger() * depth
            else:
                total += dfs(ni.getList(), depth + 1)
        return total
    return dfs(nested_list, 1)

def answer_count_by_depth(nested_list):
    counts = {}
    def dfs(items, depth):
        for ni in items:
            if ni.isInteger():
                counts[depth] = counts.get(depth, 0) + 1
            else:
                dfs(ni.getList(), depth + 1)
    dfs(nested_list, 1)
    return counts

def answer_flatten(nested_list):
    out = []
    def dfs(items):
        for ni in items:
            if ni.isInteger():
                out.append(ni.getInteger())
            else:
                dfs(ni.getList())
    dfs(nested_list)
    return out

def answer_depth_sum_inverse_dfs(nested_list):
    def get_max_depth(items, depth):
        best = 0
        for ni in items:
            if ni.isInteger():
                best = max(best, depth)
            else:
                best = max(best, get_max_depth(ni.getList(), depth + 1))
        return best
    max_d = get_max_depth(nested_list, 1)
    def dfs(items, depth):
        total = 0
        for ni in items:
            if ni.isInteger():
                total += ni.getInteger() * (max_d - depth + 1)
            else:
                total += dfs(ni.getList(), depth + 1)
        return total
    return dfs(nested_list, 1)

assert answer_depth_sum(build([2, [3, [4, 5]], [[1]]])) == 38
assert show(build([[], 1])) == "[[],1]" and len(build([[1, 2], 3])) == 2
assert answer_depth_sum(build([[1, 1], 2, [1, 1]])) == 10 and answer_depth_sum(build([1, [4, [6]]])) == 27
assert answer_count_by_depth(build([[1, 1], 2, [1, 1]])) == {1: 1, 2: 4}
assert answer_count_by_depth(build([1, [4, [6]]])) == {1: 1, 2: 1, 3: 1}
assert answer_flatten(build([[1, 1], 2, [1, 1]])) == [1, 1, 2, 1, 1] and answer_flatten(build([1, [4, [6]]])) == [1, 4, 6]
assert answer_depth_sum_inverse_dfs(build([[1, 1], 2, [1, 1]])) == 8
assert answer_depth_sum_inverse_dfs(build([1, [4, [6]]])) == 17
assert depth_sum_inverse(build([1, [[]]])) == 1    # tầng chỉ có list rỗng không làm tăng trọng số
random.seed(364)
for _ in range(300):
    data = random_nested(5) or [0]
    nl = build(data)
    assert answer_depth_sum_inverse_dfs(nl) == depth_sum_inverse(nl), data
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 19. Template nhớ nhanh + checklist submit

```python
class Solution:
    def depthSum(self, nestedList: List[NestedInteger]) -> int:
        def dfs(items, depth):
            total = 0
            for ni in items:
                if ni.isInteger():
                    total += ni.getInteger() * depth
                else:
                    total += dfs(ni.getList(), depth + 1)
            return total
        return dfs(nestedList, 1)
```

**Template DFS mang trạng thái theo tầng:** `dfs(node, state)` → xử lý node lá với `state` → gọi con với `state` cập nhật (`depth + 1`) → cộng/gộp kết quả con.

**Checklist trước khi submit:**
- [ ] Độ sâu tầng ngoài cùng là **1**?
- [ ] Hỏi `isInteger()` trước khi `getInteger()` / `getList()`?
- [ ] `depth` là tham số (không phải biến chung quên giảm)?
- [ ] BFS (nếu dùng) có `for _ in range(len(queue))` để tách tầng?
- [ ] Không định nghĩa lại `NestedInteger` khi nộp; chữ ký `depthSum(self, nestedList)` đúng?